# Healthcare Triage - Goal-Oriented Multi-Agent Flow (CrewAI)

---

### Problem Statement

In hospitals and emergency departments, medical staff must quickly assess patients and determine treatment priority.
Manual triage can be inconsistent due to fatigue, workload, or limited medical information.

A system is needed that can:
- Analyse patient symptoms from free-text input
- Assess clinical severity
- Assign a care priority level
- Adapt decisions based on whether the patient is stable or critical

---

### Approach - Goal-Oriented Multi-Agent System

This notebook uses **CrewAI** to build a pipeline of specialised AI agents.
Each agent has a distinct *role*, *goal*, and *backstory*, and they collaborate to triage a patient.

```
Patient Input
     |
     v
[  Triage Agent       ]  <- classifies severity (Mild/Moderate/Severe/Emergency)
     |
     v
[  Care Planning Agent]  <- builds evidence-based treatment plan
     |
     v
[  Decision Agent     ]  <- picks Reactive vs Deliberate care mode
     |
     v
[  Monitoring Agent   ]  <- watches for deterioration, triggers ESCALATE if needed
```

**`{patient_details}` is injected dynamically at runtime into every task - no code changes needed per patient.**


---
## Step 0 - Installation

Run the cell below **once** to install the required packages.

| Package | Purpose |
|---------|---------|
| `crewai` | Multi-agent orchestration framework |
| `langchain-openai` | LLM connector for OpenAI models |
| `python-dotenv` | Loads API keys from a `.env` file |

> **Tip:** These three packages are all you need - no extra langchain_community required.
> Always pin or use compatible versions to avoid dependency conflicts.


In [ ]:
# Uncomment and run once to install all required packages.
# These versions are tested to work together without conflicts.
# !pip install crewai langchain-openai python-dotenv

---
## Step 1 - Imports and LLM Initialisation

Here we:
1. Suppress noisy warnings for cleaner notebook output.
2. Load the **OpenAI API key** from a `.env` file (never hard-code secrets!).
3. Create a shared **LLM object** (`gpt-4o-mini`) that every agent will use.

> `temperature=0` makes responses deterministic - ideal for medical decision logic
> where you want consistent, repeatable outputs rather than creative variation.


In [ ]:
# Suppress verbose runtime warnings (e.g. from OpenTelemetry / telemetry tracers)
import warnings
warnings.filterwarnings('ignore')

import os
from langchain_openai import ChatOpenAI   # OpenAI LLM connector for CrewAI agents
from dotenv import load_dotenv            # Reads .env file for OPENAI_API_KEY

# Load OPENAI_API_KEY (and any other secrets) from a local .env file
# Create a .env file in this folder with: OPENAI_API_KEY=sk-...
load_dotenv()

# Shared LLM instance - temperature=0 ensures consistent, reproducible medical responses
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM ready:", llm.model_name)

---
## Step 2 - Import CrewAI Building Blocks

CrewAI has three core classes you need to know:

| Class | What it represents |
|-------|-------------------|
| `Agent` | An AI worker with a professional role, goal, and backstory |
| `Task` | A concrete job assigned to one agent (supports `{placeholder}` inputs) |
| `Crew` | The orchestrator that sequences agents and tasks into a pipeline |

Think of it like a hospital: **Crew** is the hospital, **Agents** are the specialists,
and **Tasks** are the procedures each specialist carries out on the patient.


In [ ]:
# Import the three core CrewAI primitives
from crewai import Agent, Task, Crew

---
## Step 3 - Define the Four Triage Agents

Each agent is built with four key parameters:

| Parameter | Purpose |
|-----------|---------|
| `role` | Job title - anchors the LLM to a professional identity |
| `goal` | The agent's primary objective for this run |
| `backstory` | Extra context that sharpens the LLM's medical persona |
| `llm` | The language model powering this agent |

**Why give agents a backstory?**
LLMs perform significantly better when given a clear professional identity.
A "Triage Specialist trained in ESI protocols" answers very differently from a generic assistant.


In [ ]:
# ========================= AGENTS =========================

# Agent 1: Receives raw patient text and performs clinical symptom classification
triage_agent = Agent(
    role="Triage Specialist",
    goal="Classify patient urgency based on symptoms and reported condition.",
    backstory=(
        "You are trained in medical emergency response and follow triage protocols "
        "like ESI (Emergency Severity Index) and CTAS (Canadian Triage and Acuity Scale)."
    ),
    llm=llm,
    verbose=True   # Set to False to suppress step-by-step agent reasoning in output
)

# Agent 2: Receives triage output and creates a prioritised treatment plan
planning_agent = Agent(
    role="Care Planning Strategist",
    goal="Generate a structured plan for treatment priorities.",
    backstory=(
        "You create evidence-based care plans and ensure medical decisions follow "
        "recognised best practice frameworks."
    ),
    llm=llm,
    verbose=True
)

# Agent 3: Decides whether to use Reactive (immediate) or Deliberative (planned) care
decision_agent = Agent(
    role="Hybrid Decision Intelligence Unit",
    goal="Select between reactive emergency response or deliberate planned care.",
    backstory=(
        "When a case is critical, you respond rapidly (reactive mode). "
        "For stable cases, you perform deeper structured reasoning (deliberative mode)."
    ),
    llm=llm,
    verbose=True
)

# Agent 4: Simulates ongoing patient monitoring and flags deterioration
monitoring_agent = Agent(
    role="Patient Monitoring and Escalation Agent",
    goal="Continuously evaluate patient vitals and symptoms to adjust urgency dynamically.",
    backstory=(
        "You track patient condition in real-time and trigger escalation "
        "if health indicators deteriorate (e.g. breathing difficulty, chest pain worsening)."
    ),
    llm=llm,
    verbose=True
)

print("All 4 agents created successfully.")

---
## Step 4 - Define Tasks

Each Task tells an agent **what to do** and **what output is expected**.

**Dynamic inputs with `{}`:**
The `{patient_details}` placeholder inside task descriptions is replaced at runtime
when `crew.kickoff(inputs={"patient_details": "..."})` is called.
This means you can run the exact same code for any patient, just by changing the input string.

**Task chaining:**
By default, CrewAI passes each task's output as context to the next task.
So the Care Planning agent automatically sees what the Triage agent found - no extra code needed.


In [ ]:
# ========================= TASKS =========================

# Task 1: Symptom identification and severity classification
# {patient_details} is replaced at runtime with the actual patient description
triage_task = Task(
    description=(
        "You will evaluate the provided patient details.\n"
        "Patient Symptoms:\n\n"
        "{patient_details}\n\n"          # <-- runtime input injected here
        "Steps:\n"
        "1. Identify key symptoms.\n"
        "2. Classify severity: Mild / Moderate / Severe / Emergency.\n"
        "3. Recommend immediate next steps.\n\n"
        "Return the result in a clear structured response."
    ),
    expected_output="A structured medical triage report including severity level and guidance.",
    agent=triage_agent    # Assigned to the Triage Specialist
)

# Task 2: Evidence-based treatment plan (receives triage output as context automatically)
care_plan_task = Task(
    description=(
        "Using the triage results and patient details: {patient_details}\n\n"
        "Generate a treatment prioritization plan with 3 to 5 actionable steps.\n"
        "Ensure actions align with evidence-based emergency response standards."
    ),
    agent=planning_agent,
    expected_output="A structured, prioritised care plan with clear action steps."
)

# Task 3: Reactive vs Deliberative mode selection
decision_task = Task(
    description=(
        "Using the triage results and care plan for: {patient_details}\n\n"
        "Decide the treatment mode:\n"
        "- 'Reactive Emergency Response' for high-severity or critical cases\n"
        "- 'Deliberative Planned Care' for stable, non-critical cases\n\n"
        "Provide justification in 2 to 3 sentences."
    ),
    agent=decision_agent,
    expected_output="Treatment mode decision with clear reasoning."
)

# Task 4: Real-time monitoring - returns ESCALATE or STABLE status
monitoring_task = Task(
    description=(
        "Simulate ongoing condition monitoring for: {patient_details}\n\n"
        "If signs of deterioration appear such as:\n"
        "  - breathing difficulty\n"
        "  - loss of consciousness\n"
        "  - chest pain worsening\n"
        "Return: 'ESCALATE IMMEDIATELY'\n\n"
        "Otherwise return: 'Patient stable. Continue existing care plan.'"
    ),
    agent=monitoring_agent,
    expected_output="Monitoring result: either ESCALATE IMMEDIATELY or Patient stable status."
)

print("All 4 tasks defined.")

---
## Step 5 - Assemble the Crew (Pipeline)

`Crew` wires agents and tasks together into a **sequential pipeline**.

- Tasks run in the order you list them.
- Each agent automatically receives the previous agent's output as context.
- `verbose=True` prints the full execution trace (great for learning and debugging).

The flow is:
```
triage_task -> care_plan_task -> decision_task -> monitoring_task
```


In [ ]:
# ========================= CREW PIPELINE =========================

# Crew sequences the agents and tasks in the order provided.
# Each task's output is passed as context to the next task automatically.
triage_pipeline = Crew(
    agents=[triage_agent, planning_agent, decision_agent, monitoring_agent],
    tasks=[triage_task, care_plan_task, decision_task, monitoring_task],
    verbose=True    # Shows full agent reasoning; set to False for cleaner output
)

print("Crew pipeline assembled and ready to run.")

---
## Step 6 - Run the Triage Pipeline

Provide a plain-text patient description and call `kickoff()`.
The pipeline injects `patient_details` into all four tasks and runs them in sequence.

Change the text in `patient_details` to any of the test cases at the bottom to see how the system responds to different severity levels.


In [ ]:
# Sample patient - a non-critical, low-severity case
# This should trigger 'Deliberative Planned Care' and a 'Patient stable' monitoring result
patient_details = (
    "Patient has mild fever, sore throat, and fatigue for two days. "
    "No trouble breathing, no chest pain."
)

# kickoff() starts the sequential pipeline and injects the input into all tasks
result = triage_pipeline.kickoff(inputs={"patient_details": patient_details})

print("\n" + "=" * 60)
print("FINAL TRIAGE OUTPUT")
print("=" * 60)
print(result)

---
## Test Cases - Try It Yourself

Replace the `patient_details` string in Step 6 with any scenario below and re-run.

| # | Patient Description | Expected System Response |
|---|--------------------|-----------------------------|
| 1 | Mild fever, sore throat, fatigue (2 days) - no breathing issues | Deliberative care, patient stable |
| 2 | Dizziness, headache, blurred vision - no vomiting, off-balance | Structured plan + possible escalation warning |
| 3 | Chest pain to left arm, shortness of breath, sweating, can't stand | **Reactive emergency response** + ESCALATE |
| 4 | 6-year-old: persistent vomiting, dehydration, lethargy | Moderate-high risk, urgent care plan |
| 5 | Severe leg injury post-fall - controlled bleeding, no loss of consciousness | Rapid triage + shock monitoring |

```python
# Case 2 - Moderate
patient_details = "Patient reports dizziness, headache, and blurred vision. No vomiting, speaking normally, but off-balance."

# Case 3 - EMERGENCY
patient_details = "Patient experiencing chest pain radiating to the left arm, shortness of breath, and sweating. Cannot stand without assistance."

# Case 4 - Paediatric
patient_details = "A 6-year-old with persistent vomiting, dehydration signs, and lethargy. No trauma or fever, but unable to keep liquids down."

# Case 5 - Trauma
patient_details = "Patient has severe leg injury after a fall. Bleeding controlled but intense pain, swelling, and limited movement. No loss of consciousness."
```
